In [1]:
import re
import os

from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.embeddings import Embeddings
from langchain_community.document_loaders import WebBaseLoader
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_ollama import ChatOllama
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv
from langsmith import traceable


c:\Users\Vyacheslave\Desktop\AI-assistant\rag_langchain_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


## RAG

### Индексация

In [2]:
# Очистка текста (аналогично вашей функции clean_text)
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,!?-]', '', text)
    text = re.sub(r'&#\d+;', '', text)
    return text.strip()

In [3]:
# Класс для добавления префиксов к запросам и документам
class PrefixedEmbeddings(Embeddings):
    def __init__(self, base, query_prefix="", doc_prefix=""):
        self.base = base
        self.query_prefix = query_prefix
        self.doc_prefix = doc_prefix

    def embed_documents(self, texts):
        texts_prefixed = [self.doc_prefix + t for t in texts]
        return self.base.embed_documents(texts_prefixed)

    def embed_query(self, text):
        return self.base.embed_query(self.query_prefix + text)

### Сплит данных и перевод их в embedding

In [4]:
# Загрузка данных с сайта
urls = ["https://smartklimat74.ru/kondicionirovanie.html",
        "https://smartklimat74.ru/katalog3.html",
        "https://smartklimat74.ru/katalog2.html",
        "https://smartklimat74.ru/uslugi.html",
        "https://smartklimat74.ru/servisnyy-centr.html"]
loader = WebBaseLoader(urls)
web_docs = loader.load()

# Преобразование и очистка данных
docs = []
for doc in web_docs:
    content = clean_text(doc.page_content)
    metadata = {"source": doc.metadata["source"]}
    docs.append(Document(page_content=content, metadata=metadata))

# Разделение на чанки
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=100)
splits = text_splitter.split_documents(docs)


In [5]:
print(splits[0].page_content.replace('\n', ' '))
print("------------------------")
print(splits[1].page_content.replace('\n', ' '))
print("------------------------")

Кондиционирование - Продажа, монтаж. ремонт климатического оборудования Главное меню Close Назад АкцииСпецпредложения Кондиционирование Бытовые сплит-системы Ballu Dahaci Electrolux Haier Hisense Lessar LG Midea QuattroClima ROYAL CLIMA SHUFT Tosot XIGMA Полупромышленные сплит-системы Микроклимат Приточные очистители воздуха Тепловое оборудование Тепловентиляторы Тепловые завесы компактные Услуги Сервисный центр Close Назад ДоставкаКонтактыО компании Бесплатная доставка до подъезда по г. Челябинск, г. Копейск. Подробности x СмартКлимат74 ДоставкаКонтактыО компании  Отложенные 0 шт. В сравнении 0 шт.  Кабинет 0 p 7 919 333 22 11 Отложенные 0 шт. Сравнить 0 шт. АкцииСпецпредложения Кондиционирование Бытовые сплит-системы Ballu Dahaci Electrolux Haier Hisense Lessar LG Midea QuattroClima ROYAL CLIMA SHUFT Tosot XIGMA Полупромышленные сплит-системы Микроклимат Приточные очистители воздуха Тепловое оборудование Тепловентиляторы Тепловые завесы компактные Услуги Сервисный центр Каталог Весь 

In [6]:
# Создание эмбеддингов
base_embeddings = HuggingFaceEmbeddings(model_name="ai-forever/ru-en-RoSBERTa")
embeddings = PrefixedEmbeddings(base_embeddings, query_prefix="search_query: ", doc_prefix="search_document: ")

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 11441.36it/s]
RobertaModel LOAD REPORT from: ai-forever/ru-en-RoSBERTa
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Создание ВБД-шки

In [7]:
# Директория, где будет ВБД
persist_directory = "./web_chroma_db"

if os.path.isdir(persist_directory):
    # Индекс уже есть – просто загружаем
    vectorstore = Chroma(
        embedding_function=embeddings,
        persist_directory=persist_directory,
    )
else: 
    # Создание или загрузка векторной базы  
    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,
        persist_directory=persist_directory,
    )

C:\Users\Vyacheslave\AppData\Local\Temp\ipykernel_14728\715859590.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


### Инициализуция MMR-ретривера

In [8]:
# 1. Базовый retriever с MMR (разнообразные документы)
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",  # вместо простого similarity
    search_kwargs={
        "k": 8,          # сколько документов вернуть в итоге
        "fetch_k": 32,   # из скольких кандидатов выбирать (больше = разнообразнее)
        # при желании можно добавить lambda_mult для тонкой настройки
        # lambda_mult – это параметр MMR, который задаёт баланс между релевантностью документов запросу и их разнообразием: чем ближе значение к 1, тем сильнее приоритет близости к запросу, чем ближе к 0 – тем важнее разнообразие результатов.
        # "lambda_mult": 0.8,
    },
)

### Шаблон промта

In [9]:
# Промпт с поддержкой истории
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Ты специалист в области систем кондиционирования. "
        "Используй следующую информацию для ответа на вопрос. "
        "Отвечай только на русском. "
        "Отвечай только на те вопросы, которые связаны с областью твоей специальности. "
        "Если вопрос не относится к твоей специальности, вежливо уведоми об этом пользователя. "
        "Если не можешь найти ответ на вопрос, вежливо уведоми пользователя о том, что не обладаешь знаниями об запрашиваемой информации. "
        "Если пользователя интересует какой-то товар, порекомендуй несколько вариантов, указывай бренд, модель и кратко о характеристиках, но постарайся выделить лучший, исходя из вопроса пользователя."
    ),
    MessagesPlaceholder(variable_name="history"),
    (
        "human",
        "Контекст: {context}\n\nВопрос: {question}"
    ),
])

### Подключаем LLM для генерации

In [10]:
# Модель Ollama
llm = ChatOllama(
    model="gemma2:9b",
    temperature=0.4,
    max_tokens=512,
    top_p=0.9,
)

max_chars = 8000

In [11]:
def format_docs(docs):
    formatted = []
    total_len = 0
    for doc in docs:
        source = doc.metadata.get("source", "unknown_source")
        text = doc.page_content.strip()
        block = f"Source: {source}\n{text}"
        if total_len + len(block) > max_chars:
            break
        formatted.append(block)
        total_len += len(block)
    return "\n\n---\n\n".join(formatted)

def ensure_context(input_dict: dict) -> dict:
    context = input_dict.get("context", "").strip()
    if not context:
        input_dict["context"] = (
            "Контекст пуст: ретривер не нашёл ни одного подходящего фрагмента. "
            "Если ответ важен, лучше явно сказать пользователю об этом."
        )
    return input_dict

# История диалога вручную
history = []

### RAG-Цепь

In [12]:
# Цепочка RAG с историей
def get_rag_chain(history):
    return (
        {
            "context": lambda d: d.get("context", ""),
            "question": lambda d: d.get("question", ""),
            "history": lambda _: history,
        }
        | RunnableLambda(ensure_context)
        | prompt
        | llm
        | StrOutputParser()
    )

In [13]:
@traceable(name="AirCond_answer_question")
def answer_question(question: str, context: str, history: list) -> str:
    rag_chain = get_rag_chain(history)
    inputs = {
        "question": question,
        "context": context,
    }
    answer = rag_chain.invoke(inputs)
    history.extend([HumanMessage(content=question), AIMessage(content=answer)])
    return answer

## Корректировка вопроса пользователя

### Инициализация LLM и промптов

In [14]:
question_rewrite_llm = ChatOpenAI(
    api_key="None",
    base_url="http://127.0.0.1:11434/v1",
    model="mistral-nemo:latest",
    temperature=0.2,
    max_tokens=512,
    top_p=0.9,
)

In [15]:
question_rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Ты AI-агент, который подготавливает пользовательские вопросы для RAG-системы, отвечающей на вопросы по сервисам, продуктам и услугам компании SmartKlimat74 на основе её публичных веб-ресурсов (сайт, FAQ, справочные материалы, каталоги оборудования, инструкции)."
        "Твоя задача:"
        "1) Точно определить суть вопроса пользователя."
        "2) Оценить, насколько формулировка подходит для семантического поиска по документации и материалам SmartKlimat74."
        "3) Если вопрос слишком общий, разговорный, содержит неясные местоимения (например, 'это', 'там', 'оно') или требует уточнения, переписать его в чёткий, формальный и самодостаточный вид."
        "Правила переписывания:"
        "- Всегда явно упоминай SmartKlimat74, если это релевантно контексту вопроса."
        "- Сохраняй исходный смысл, не добавляй новых фактов или предположений."
        "- Формулируй вопрос так, чтобы по нему можно было найти точный ответ в документации, FAQ, каталогах или инструкциях компании."
        "- Используй нейтральный, технический стиль без разговорных выражений и эмоциональной окраски."
        "Если исходный вопрос уже оптимально сформулирован для поиска — верни его без изменений."
        "Отвечай ТОЛЬКО итоговой формулировкой вопроса, без кавычек, пояснений и комментариев."
    ),
    ("human", "{question}")
])

In [16]:
question_rewrite_chain = (
    question_rewrite_prompt
    | question_rewrite_llm
    | StrOutputParser()
)


In [17]:
def rewrite_question_if_needed(question: str) -> str:
    rewritten = question_rewrite_chain.invoke({"question": question}).strip()
    return rewritten

### Инициализация диалога

In [ ]:
print("Здравсвуйте! Я помощник по системам кондиционирования. Задайте свой вопрос или напишите 'выход', чтобы завершить диалог.")
while True:
    user_input = input("Вы: ")
    if user_input.lower() in ["выход", "quit", "exit"]:
        print("Диалог завершён. До свидания!")
        break

    q = rewrite_question_if_needed(user_input)
    retrieved_docs = mmr_retriever.invoke(q)
    ctx = format_docs(retrieved_docs)

    answer = answer_question(question=q, context=ctx, history=history)
    print(f"Бот: {answer}\n")


Привет! Я помощник по системам кондиционирования. Задайте свой вопрос или напишите 'выход', чтобы завершить диалог.
Бот: Из предоставленной информации я не могу сказать, какие именно сервисы, продукты или услуги предлагает компания SmartKlimat74.  Ссылки, которые вы предоставили, ведут на страницы, которые не найдены. 

Чтобы получить эту информацию, я рекомендую посетить официальный сайт компании SmartKlimat74. 


Бот: К сожалению, я не могу получить доступ к внешним веб-сайтам и просматривать их содержимое, включая страницы SmartKlimat74. Поэтому я не знаю, какие именно продукты и услуги они предлагают.  

Попробуйте посетить сайт компании напрямую, чтобы узнать больше о их продуктах и услугах. 


Диалог завершён. До свидания!
